# Web Crawler Demo: linktrace with Callbacks (Memory-Efficient)

This notebook demonstrates the linktrace Spider using callbacks with `accumulate_results=False` to process results as they're crawled without accumulating them in memory.

**Comparison:** Compare memory usage with `crawl_cnn.ipynb` which accumulates all results in memory.

In [ ]:
import json
import logging
from collections import Counter
from datetime import datetime

import psutil

from linktrace import Spider

# Configure logging to see crawler activity
logging.basicConfig(
    level=logging.INFO, format="%(asctime)s - %(name)s - %(levelname)s - %(message)s"
)

# Get initial memory usage
process = psutil.Process()
initial_memory = process.memory_info().rss / 1024 / 1024  # MB
print(f"Initial memory: {initial_memory:.2f} MB")

## Stream URLs with Callbacks (Memory-Efficient)

Define callbacks to process and display results as they're crawled, without keeping them in memory.

In [4]:
# Track crawl statistics
crawl_stats = {
    "total_pages": 0,
    "total_internal_links": 0,
    "total_external_links": 0,
    "failed_urls": [],
}

# Store first 10 URLs for display
urls_crawled = []


def on_page_crawled(doc):
    """Called after each page is crawled - process and discard."""
    crawl_stats["total_pages"] += 1
    crawl_stats["total_internal_links"] += len(doc.internal_links)
    crawl_stats["total_external_links"] += len(doc.external_links)

    # Store first 10 URLs for display
    if len(urls_crawled) < 10:
        urls_crawled.append(
            {
                "url": doc.url,
                "title": doc.title[:60],
                "status": doc.status_code,
                "internal_links": len(doc.internal_links),
                "external_links": len(doc.external_links),
            }
        )

    # Print progress every 10 pages
    if crawl_stats["total_pages"] % 10 == 0:
        current_memory = process.memory_info().rss / 1024 / 1024
        print(
            f"  Crawled {crawl_stats['total_pages']} pages | "
            f"Memory: {current_memory:.2f} MB | "
            f"Last: {doc.url}"
        )

    # Don't return anything - results are discarded
    return None


def on_error(url, exception):
    """Called when a page fails to crawl."""
    crawl_stats["failed_urls"].append({"url": url, "error": str(exception)})
    print(f"  ❌ Failed: {url}")


def on_crawl_complete():
    """Called when crawl finishes."""
    print(f"\n✓ Crawl complete!")


# Run spider with callbacks - accumulate_results=False means no memory buildup
spider = Spider(
    start_url="https://www.cnn.com",
    max_depth=1,
    on_page_crawled=on_page_crawled,
    on_error=on_error,
    on_crawl_complete=on_crawl_complete,
    accumulate_results=False,  # KEY: Don't accumulate in memory
    show_progress=True,
)

print(f"Starting streaming crawl with max_depth=3...\n")
result = await spider.run_async()

print(f"\nResult list: {len(result)} (empty because accumulate_results=False)")

2026-06-08 16:36:27,093 WebCrawler.Spider DEBUG    Spider initialized: strategy=BFS, max_depth=1
2026-06-08 16:36:27,093 - WebCrawler.Spider - DEBUG - Spider initialized: strategy=BFS, max_depth=1


Starting streaming crawl with max_depth=3...



Crawling: 0 URLs [00:00, ? URLs/s]2026-06-08 16:36:27,419 WebCrawler.Spider DEBUG    Fetching https://www.cnn.com (attempt 1)
2026-06-08 16:36:27,419 WebCrawler.Spider DEBUG    Fetching https://www.cnn.com (attempt 1)
2026-06-08 16:36:27,419 - WebCrawler.Spider - DEBUG - Fetching https://www.cnn.com (attempt 1)
2026-06-08 16:36:27,498 WebCrawler.Spider INFO     Visited: https://www.cnn.com (Total Visited: 1)
2026-06-08 16:36:27,498 WebCrawler.Spider INFO     Visited: https://www.cnn.com (Total Visited: 1)
2026-06-08 16:36:27,498 - WebCrawler.Spider - INFO - Visited: https://www.cnn.com (Total Visited: 1)
Crawling: 1 URLs [00:00,  2.49 URLs/s, visited=1, pending=0]2026-06-08 16:36:27,538 WebCrawler.Spider DEBUG    Fetching https://www.cnn.com/us (attempt 1)
2026-06-08 16:36:27,538 WebCrawler.Spider DEBUG    Fetching https://www.cnn.com/us (attempt 1)
2026-06-08 16:36:27,538 - WebCrawler.Spider - DEBUG - Fetching https://www.cnn.com/us (attempt 1)
2026-06-08 16:36:27,546 WebCrawler.Spide

  Crawled 10 pages | Memory: 852.97 MB | Last: https://www.cnn.com/health


2026-06-08 16:36:28,130 WebCrawler.Spider INFO     Visited: https://www.cnn.com/science (Total Visited: 16)
2026-06-08 16:36:28,130 - WebCrawler.Spider - INFO - Visited: https://www.cnn.com/science (Total Visited: 16)
Crawling: 16 URLs [00:01, 18.86 URLs/s, visited=16, pending=7798]2026-06-08 16:36:28,167 WebCrawler.Spider INFO     Visited: https://www.cnn.com/weather (Total Visited: 17)
2026-06-08 16:36:28,167 WebCrawler.Spider INFO     Visited: https://www.cnn.com/weather (Total Visited: 17)
2026-06-08 16:36:28,167 - WebCrawler.Spider - INFO - Visited: https://www.cnn.com/weather (Total Visited: 17)
Crawling: 17 URLs [00:01, 18.86 URLs/s, visited=17, pending=8721]2026-06-08 16:36:28,194 WebCrawler.Spider INFO     Visited: https://www.cnn.com/sport/fifa-world-cup (Total Visited: 18)
2026-06-08 16:36:28,194 WebCrawler.Spider INFO     Visited: https://www.cnn.com/sport/fifa-world-cup (Total Visited: 18)
2026-06-08 16:36:28,194 - WebCrawler.Spider - INFO - Visited: https://www.cnn.com/sp

...

  Crawled 180 pages | Memory: 2545.25 MB | Last: https://www.cnn.com/2026/06/08/world/video/lebanon-key-to-israel-iran-war-future-digvid-vrtc


2026-06-08 16:36:37,019 WebCrawler.Spider INFO     Visited: https://www.cnn.com/2026/06/07/world/video/iran-fires-missiles-northern-israel-diamond-vrtc (Total Visited: 184)
2026-06-08 16:36:37,019 WebCrawler.Spider INFO     Visited: https://www.cnn.com/2026/06/07/world/video/iran-fires-missiles-northern-israel-diamond-vrtc (Total Visited: 184)
2026-06-08 16:36:37,019 - WebCrawler.Spider - INFO - Visited: https://www.cnn.com/2026/06/07/world/video/iran-fires-missiles-northern-israel-diamond-vrtc (Total Visited: 184)
Crawling: 184 URLs [00:09, 16.36 URLs/s, visited=184, pending=362395]2026-06-08 16:36:37,282 WebCrawler.Spider DEBUG    Fetching https://www.cnn.com/2026/06/07/world/video/100-days-iran-war-digvid-vrtc (attempt 1)
2026-06-08 16:36:37,282 WebCrawler.Spider DEBUG    Fetching https://www.cnn.com/2026/06/07/world/video/100-days-iran-war-digvid-vrtc (attempt 1)
2026-06-08 16:36:37,282 - WebCrawler.Spider - DEBUG - Fetching https://www.cnn.com/2026/06/07/world/video/100-days-iran-

  Crawled 190 pages | Memory: 3133.14 MB | Last: https://www.cnn.com/2026/06/07/politics/video/mark-warner-bill-pulte-security-risk-digvid-vrtc


2026-06-08 16:36:37,808 WebCrawler.Spider INFO     Visited: https://www.cnn.com/subscription?source=sub_web_footerlink-link (Total Visited: 194)
2026-06-08 16:36:37,808 WebCrawler.Spider INFO     Visited: https://www.cnn.com/subscription?source=sub_web_footerlink-link (Total Visited: 194)
2026-06-08 16:36:37,808 - WebCrawler.Spider - INFO - Visited: https://www.cnn.com/subscription?source=sub_web_footerlink-link (Total Visited: 194)
Crawling: 194 URLs [00:10, 15.21 URLs/s, visited=194, pending=393850]2026-06-08 16:36:37,860 WebCrawler.Spider INFO     Visited: https://www.cnn.com/accessibility (Total Visited: 195)
2026-06-08 16:36:37,860 WebCrawler.Spider INFO     Visited: https://www.cnn.com/accessibility (Total Visited: 195)
2026-06-08 16:36:37,860 - WebCrawler.Spider - INFO - Visited: https://www.cnn.com/accessibility (Total Visited: 195)
Crawling: 195 URLs [00:10, 15.90 URLs/s, visited=195, pending=396991]2026-06-08 16:36:37,925 WebCrawler.Spider INFO     Visited: https://www.cnn.co


✓ Crawl complete!

Result list: 0 (empty because accumulate_results=False)


## Crawl Statistics

In [5]:
# Calculate final memory usage
final_memory = process.memory_info().rss / 1024 / 1024
memory_change = final_memory - initial_memory

print("=" * 70)
print("CRAWL STATISTICS (Streaming with Callbacks)")
print("=" * 70)
print(f"Total pages crawled: {crawl_stats['total_pages']}")
print(f"Total internal links found: {crawl_stats['total_internal_links']}")
print(f"Total external links found: {crawl_stats['total_external_links']}")
print(f"Failed URLs: {len(crawl_stats['failed_urls'])}")
print()
print(f"Initial memory: {initial_memory:.2f} MB")
print(f"Final memory: {final_memory:.2f} MB")
print(f"Memory change: {memory_change:+.2f} MB")
print(f"Avg memory per page: {memory_change / crawl_stats['total_pages']:.4f} MB")
print()
print(
    f"Avg links per page: "
    f"{(crawl_stats['total_internal_links'] + crawl_stats['total_external_links']) / crawl_stats['total_pages']:.1f}"
)

CRAWL STATISTICS (Streaming with Callbacks)
Total pages crawled: 199
Total internal links found: 431448
Total external links found: 38924
Failed URLs: 0

Initial memory: 545.08 MB
Final memory: 3206.66 MB
Memory change: +2661.58 MB
Avg memory per page: 13.3748 MB

Avg links per page: 2363.7


## First 10 URLs Crawled

In [6]:
print(f"{'URL':<50} {'Title':<20} {'Status':<8} {'Links':<8}")
print("-" * 90)

for item in urls_crawled:
    url_short = item["url"][:49]
    title_short = item["title"][:19]
    total_links = item["internal_links"] + item["external_links"]
    print(
        f"{url_short:<50} {title_short:<20} "
        f"{item['status']:<8} {total_links:<8}"
    )

URL                                                Title                Status   Links   
------------------------------------------------------------------------------------------
https://www.cnn.com                                Breaking News, Late  200      221     
https://www.cnn.com/us                             US | CNN             200      257     
https://www.cnn.com/world                          World news - breaki  200      343     
https://www.cnn.com/business                       Business News - Lat  200      395     
https://www.cnn.com/style                          CNN Style - Fashion  200      432     
https://www.cnn.com/entertainment                  Entertainment | CNN  200      465     
https://www.cnn.com/cnn-underscored                                     403      0       
https://www.cnn.com/politics                       Politics | CNN Poli  200      518     
https://www.cnn.com/travel                         CNN Travel | Global  200      559     
https://w

## Failed URLs (if any)

In [7]:
if crawl_stats["failed_urls"]:
    print(f"Failed URLs ({len(crawl_stats['failed_urls'])}):")
    for failed in crawl_stats["failed_urls"]:
        print(f"  • {failed['url']}")
        print(f"    Error: {failed['error'][:60]}...")
else:
    print("No failed URLs!")

No failed URLs!


## Comparison: Streaming vs Accumulating

**This notebook (Streaming):**
- ✅ Results processed as pages are crawled
- ✅ No results kept in memory
- ✅ Predictable, minimal memory growth
- ✅ Perfect for large crawls

**crawl_cnn.ipynb (Accumulating):**
- ❌ All pages accumulated in `documents` list
- ❌ Memory grows linearly with page count
- ❌ Can cause OOM on very large crawls
- ✅ Convenient for analysis at end

**For large crawls (1000+ pages):** Use streaming callbacks (`accumulate_results=False`)